In [1]:
import os
import logging
import argparse
import datetime
import time
from dotenv import load_dotenv

import pandas as pd
from sqlalchemy import text
import numpy as np

import sys
import os
load_dotenv("/Users/huggingfaceit/code/SoccerBetMLOptimizer/.env")
load_dotenv("/Users/huggingfaceit/code/SoccerBetMLOptimizer/secrets.env")


# Chemin vers le dossier qui contient app/
sys.path.append(os.path.abspath("../../pipelines"))
sys.path.append(os.path.abspath("../../pipelines/app/pipeline__RSF_PR_LR"))

# Maintenant tu peux importer
from utils.test_model_and_infer import test_model_and_infer
from utils.insert_results_to_db import insert_results_to_db
from feature_eng.format_df import merge_sofifa_fbref_results, format_sofifa_fbref_data, add_signals
from app._config import DB_TN_FBREF_RESULTS, DB_TN_SOFIFA_TEAMS_STATS, DB_TN_MODELS_RESULTS
from app._config import engine


In [2]:

date_stop = None
start_pipeline = time.time()


#### Connection to the database and retrieve data ####
start_data_retrieval = time.time()
with engine.connect() as connection:
    print("Database connection established")
    print(f"DB_HOST: {engine.url.host}")
    print(f"DB_PORT: {engine.url.port}")
    print(f"DB_NAME: {engine.url.database}")

    query_fbref_results = text(f"SELECT * FROM {DB_TN_FBREF_RESULTS}")
    fbref_results_df = pd.read_sql(query_fbref_results, connection)

    query_sofifa_team_stats = text(f'SELECT * FROM {DB_TN_SOFIFA_TEAMS_STATS}')
    sofifa_teams_stats_df = pd.read_sql(query_sofifa_team_stats, connection)
    print(f"Data retrieved from database successfully in {time.time() - start_data_retrieval} seconds")



Database connection established
DB_HOST: localhost
DB_PORT: 5432
DB_NAME: optimsportbets-db
Data retrieved from database successfully in 1.6000950336456299 seconds


In [3]:
fbref_results_df.sort_values(by=["date"], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,venue,referee,match_report,notes,index,away_g,home_g,away_sat,home_sat,datetime_insert
89260,None,UEFA Champions League,2526,2025-07-30 Panathinaikos-Rangers,Second qualifying round,NaN,Wed,2025-07-30,21:00:00,Panathinaikos,...,None,None,None,Leg 2 of 2,9770450,NaN,NaN,NaN,NaN,2025-07-06 00:02:25.195028
89262,None,UEFA Champions League,2526,2025-07-30 Servette FC-Viktoria Plzeň,Second qualifying round,NaN,Wed,2025-07-30,21:00:00,Servette FC,...,Stade de Genève,None,None,Leg 2 of 2,47832,NaN,NaN,NaN,NaN,2025-07-06 00:02:25.195028
89261,None,UEFA Champions League,2526,2025-07-30 RB Salzburg-Brann,Second qualifying round,NaN,Wed,2025-07-30,20:45:00,RB Salzburg,...,None,None,None,Leg 2 of 2,11306221,NaN,NaN,NaN,NaN,2025-07-06 00:02:25.195028
89259,None,UEFA Champions League,2526,2025-07-30 Maccabi Tel Aviv-Pafos FC,Second qualifying round,NaN,Wed,2025-07-30,20:00:00,Maccabi Tel Aviv,...,TSC Arena,None,None,Leg 2 of 2,3661407,NaN,NaN,NaN,NaN,2025-07-06 00:02:25.195028
89258,None,UEFA Champions League,2526,2025-07-23 Brann-RB Salzburg,Second qualifying round,NaN,Wed,2025-07-23,19:00:00,Brann,...,None,None,None,Leg 1 of 2,14840637,NaN,NaN,NaN,NaN,2025-07-06 00:02:25.195028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,c4630ad0,ENG-Premier League,2425,1924-08-30 Nott'ham Forest-Arsenal,None,1.0,Sat,1924-08-30,None,Nott'ham Forest,...,None,None,/en/matches/c4630ad0/Nottingham-Forest-Arsenal...,None,14930337,2.0,0.0,NaN,NaN,2024-09-24 09:03:40.882027
4,5f803c50,ENG-Premier League,2425,1924-08-30 Bury-Manchester City,None,1.0,Sat,1924-08-30,None,Bury,...,None,None,/en/matches/5f803c50/Bury-Manchester-City-Augu...,None,15921462,2.0,0.0,NaN,NaN,2024-09-24 09:03:40.882027
3,8b4a0b4c,ENG-Premier League,2425,1924-08-30 Leeds United-Sunderland,None,1.0,Sat,1924-08-30,None,Leeds United,...,None,None,/en/matches/8b4a0b4c/Leeds-United-Sunderland-A...,None,6932190,1.0,1.0,NaN,NaN,2024-09-24 09:03:40.882027
2,be6f0ecf,ENG-Premier League,2425,1924-08-30 Liverpool-Aston Villa,None,1.0,Sat,1924-08-30,None,Liverpool,...,None,None,/en/matches/be6f0ecf/Liverpool-Aston-Villa-Aug...,None,12263104,4.0,2.0,NaN,NaN,2024-09-24 09:03:40.882027


In [41]:
mapping = {
    "VfL Bochum 1848": "Bochum",
    "Tottenham Hotspur": "Tottenham",
    "Paris Saint-Germain": "Paris S-G",
    "FC Köln": "Köln",
    "Real Zaragoza": "Zaragoza",
    "Wolverhampton Wanderers": "Wolves",
    "Sheffield United": "Sheffield Utd",
    "Amiens SC": "Amiens",
    "FSV Mainz 05": "Mainz 05",
    "Paderborn": "Paderborn 07",
    "Bolton Wanderers": "Bolton",
    "Huddersfield Town": "Huddersfield",
    "Olympique de Marseille": "Marseille",
    "LOSC Lille": "Lille",
    "Grenoble Foot 38": "Grenoble",
    "Racing Santander": "Racing Sant",
    "Eintracht Frankfurt": "Eint Frankfurt",
    "Fortuna Düsseldorf": "Düsseldorf",
    "Queens Park Rangers": "QPR",
    "SC Freiburg": "Freiburg",
    "DSC Arminia Bielefeld": "Arminia",
    "Republic of Ireland": "Rep. of Ireland",
    "Evian TG": "Evian",
    "FC Barcelona": "Barcelona",
    "Brighton & Hove Albion": "Brighton",
    "Deportivo La Coruña": "La Coruña",
    "Angers SCO": "Angers",
    "West Ham United": "West Ham",
    "VfL Wolfsburg": "Wolfsburg",
    "FC Augsburg": "Augsburg",
    "India": "India",
    "Bari 1908": "Bari",
    "Czech Republic": "Czechia",
    "Nottingham Forest": "Nott'ham Forest",
    "Newcastle United": "Newcastle Utd",
    "Borussia Dortmund": "Dortmund",
    "AFC Bournemouth": "Bournemouth",
    "Iran": "IR Iran",
    "Borussia Mönchengladbach": "Gladbach",
    "Olympique Lyonnais": "Lyon",
    "Venezuela": "Venezuela",
    "TSG Hoffenheim": "Hoffenheim",
    "SD Eibar": "Eibar",
    "West Bromwich Albion": "West Brom",
    "VfB Stuttgart": "Stuttgart",
    "Arles": "Arles-Avignon",
    "Stade de Reims": "Reims",
    "Stade Brestois 29": "Brest",
    "Real Valladolid": "Valladolid",
    "Clermont": "Clermont Foot",
    "FC Union Berlin": "Union Berlin",
    "Manchester United": "Manchester Utd",
    "Deportivo Alavés": "Alavés",
    "Celta de Vigo": "Celta Vigo",
    "Bayer 04 Leverkusen": "Leverkusen",
    "FC Bayern München": "Bayern Munich",
    "SpVgg Greuther Fürth": "Greuther Fürth",
    "Ingolstadt": "Ingolstadt 04",
    "Eintracht Braunschweig": "Braunschweig",
    "Blackburn Rovers": "Blackburn",
    "Stade de Reims ": "Reims",
}

team_col='team'
update_col='update'
home_team_col='home_team'
away_team_col='away_team'
date_col='date'

# Appliquer le mapping au DataFrame sofifa_teams_stats_df
sofifa_teams_stats_df[team_col] = sofifa_teams_stats_df[team_col].replace(mapping)
                                                                    
fbref_results_df[date_col] = pd.to_datetime(fbref_results_df[date_col])
fbref_results_df_date = fbref_results_df[fbref_results_df[date_col] >= min(sofifa_teams_stats_df[update_col])]

# Extraire la liste des équipes du DataFrame sofifa_teams_stats_df
sofifa_teams = set(sofifa_teams_stats_df[team_col].unique())

# Filtrer les lignes de fbref_results_df
fbref_df_date_filtered = fbref_results_df_date[
    (fbref_results_df_date[home_team_col].isin(sofifa_teams)) & 
    (fbref_results_df_date[away_team_col].isin(sofifa_teams))
]

# Assurance que les dates sont correctement formatées
fbref_df_date_filtered.loc[:, date_col] = pd.to_datetime(fbref_df_date_filtered[date_col])
sofifa_teams_stats_df[update_col] = pd.to_datetime(sofifa_teams_stats_df[update_col])

# Trier les dataframes pour la jointure asynchrone
fbref_df_date_filtered = fbref_df_date_filtered.sort_values(by=date_col)
sofifa_teams_stats_df = sofifa_teams_stats_df.sort_values(by=update_col)

In [42]:
fbref_results_df_date[fbref_results_df_date['league']=='FIFA Club World Cup'].sort_values(by=["date"], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,venue,referee,match_report,notes,index,away_g,home_g,away_sat,home_sat,datetime_insert
89574,None,FIFA Club World Cup,2526,2025-07-05 Real Madrid-Dortmund,Quarter-finals,NaN,Sat,2025-07-05,16:00:00,Real Madrid,...,MetLife Stadium (Neutral Site),None,None,None,10943231,NaN,NaN,NaN,NaN,2025-07-05 16:14:53.800444
89573,None,FIFA Club World Cup,2526,2025-07-05 Paris S-G-Bayern Munich,Quarter-finals,NaN,Sat,2025-07-05,12:00:00,Paris S-G,...,Mercedes-Benz Stadium (Neutral Site),None,None,None,6811435,NaN,NaN,NaN,NaN,2025-07-05 16:14:53.800444
89572,54878ae5,FIFA Club World Cup,2526,2025-07-04 Palmeiras-Chelsea,Quarter-finals,NaN,Fri,2025-07-04,21:00:00,Palmeiras,...,Lincoln Financial Field (Neutral Site),Alireza Faghani,/en/matches/54878ae5/Palmeiras-Chelsea-July-4-...,None,1701782,2.0,1.0,NaN,NaN,2025-07-05 16:14:53.800444
89571,5efdb2f3,FIFA Club World Cup,2526,2025-07-04 Fluminense-Al-Hilal,Quarter-finals,NaN,Fri,2025-07-04,15:00:00,Fluminense,...,Camping World Stadium (Neutral Site),Danny Makkelie,/en/matches/5efdb2f3/Fluminense-Al-Hilal-July-...,None,5496855,1.0,2.0,NaN,NaN,2025-07-05 16:14:53.800444
89570,7a0bf80a,FIFA Club World Cup,2526,2025-07-01 Real Madrid-Juventus,Round of 16,NaN,Tue,2025-07-01,15:00:00,Real Madrid,...,Hard Rock Stadium (Neutral Site),Szymon Marciniak,/en/matches/7a0bf80a/Real-Madrid-Juventus-July...,None,8800658,0.0,1.0,NaN,NaN,2025-07-05 16:14:53.800444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89517,416db7a8,FIFA Club World Cup,2526,2025-06-15 Paris S-G-Atlético Madrid,Group stage,1.0,Sun,2025-06-15,12:00:00,Paris S-G,...,Rose Bowl (Neutral Site),István Kovács,/en/matches/416db7a8/Paris-Saint-Germain-Atlet...,None,13140750,0.0,4.0,NaN,NaN,2025-07-05 16:14:53.800444
89516,debde3b8,FIFA Club World Cup,2526,2025-06-15 Palmeiras-Porto,Group stage,1.0,Sun,2025-06-15,18:00:00,Palmeiras,...,MetLife Stadium (Neutral Site),Said Martínez,/en/matches/debde3b8/Palmeiras-Porto-June-15-2...,None,4647536,0.0,0.0,NaN,NaN,2025-07-05 16:14:53.800444
89515,43308ee3,FIFA Club World Cup,2526,2025-06-15 Botafogo (RJ)-Seattle Sounders,Group stage,1.0,Sun,2025-06-15,19:00:00,Botafogo (RJ),...,Lumen Field (Neutral Site),Glenn Nyberg,/en/matches/43308ee3/Botafogo-RJ-Seattle-Sound...,None,4874017,1.0,2.0,NaN,NaN,2025-07-05 16:14:53.800444
89513,c446a386,FIFA Club World Cup,2526,2025-06-14 Al Ahly-Inter Miami,Group stage,1.0,Sat,2025-06-14,20:00:00,Al Ahly,...,Hard Rock Stadium (Neutral Site),Alireza Faghani,/en/matches/c446a386/Al-Ahly-Inter-Miami-June-...,None,12314746,0.0,0.0,NaN,NaN,2025-07-05 16:14:53.800444


In [43]:
sofifa_teams

{'1. FC Heidenheim 1846',
 '1. FC Union Berlin',
 '1. FSV Mainz 05',
 'AC Milan',
 'AJ Auxerre',
 'AS Monaco',
 'AS Saint-Étienne',
 'Ajaccio',
 'Alavés',
 'Albania',
 'Almería',
 'Amiens',
 'Angers',
 'Argentina',
 'Arles-Avignon',
 'Arminia',
 'Arsenal',
 'Aston Villa',
 'Atalanta',
 'Athletic Club',
 'Atlético Madrid',
 'Augsburg',
 'Australia',
 'Austria',
 'Auxerre',
 'Barcelona',
 'Bari',
 'Bastia',
 'Bayern Munich',
 'Belgium',
 'Benevento',
 'Birmingham City',
 'Blackburn',
 'Blackpool',
 'Bochum',
 'Bolivia',
 'Bologna',
 'Bolton',
 'Bordeaux',
 'Boulogne',
 'Bournemouth',
 'Braunschweig',
 'Brazil',
 'Brentford',
 'Brescia',
 'Brest',
 'Brighton',
 'Bulgaria',
 'Burnley',
 'CA Osasuna',
 'CD Leganés',
 'Caen',
 'Cagliari',
 'Cameroon',
 'Canada',
 'Cardiff City',
 'Carpi',
 'Catania',
 'Celta Vigo',
 'Cesena',
 'Chelsea',
 'Chievo',
 'Chile',
 'China PR',
 'Clermont Foot',
 'Colombia',
 'Como',
 'Costa Rica',
 'Cremonese',
 'Croatia',
 'Crotone',
 'Crystal Palace',
 'Czechia'

In [4]:
#### Data processing ####
start_data_processing = time.time()
date_stop = date_stop if date_stop else datetime.datetime.now()
fbref_results_df__sofifa_merged = merge_sofifa_fbref_results(fbref_results_df, sofifa_teams_stats_df)
fbref_results_df__sofifa_merged__data_formated = format_sofifa_fbref_data(fbref_results_df__sofifa_merged, date_stop=date_stop)
fbref_results_df__sofifa_merged__data_formated__signals_added = add_signals(fbref_results_df__sofifa_merged__data_formated, date_stop=date_stop)

rule_is_before_datetime = fbref_results_df__sofifa_merged__data_formated__signals_added["datetime"] < date_stop
fbref_results_df__sofifa_merged__data_formated__signals_added__train = fbref_results_df__sofifa_merged__data_formated__signals_added[rule_is_before_datetime]
fbref_results_df__sofifa_merged__data_formated__signals_added__infer = fbref_results_df__sofifa_merged__data_formated__signals_added[~rule_is_before_datetime]

print(f"Data processing completed successfully in {time.time() - start_data_processing} seconds")
print(f"Number of matches in train set: {fbref_results_df__sofifa_merged__data_formated__signals_added__train.shape[0]}")
print(f"Number of matches in inference set: {fbref_results_df__sofifa_merged__data_formated__signals_added__infer.shape[0]}")


Data processing completed successfully in 12.089385032653809 seconds
Number of matches in train set: 30396
Number of matches in inference set: 0


In [6]:
fbref_results_df__sofifa_merged.sort_values(by=["date"], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,away_defence_defender_line,away_defence_domestic_prestige,away_international_prestige,away_players,away_starting_xi_average_age,away_whole_team_average_age,away_fifa_edition,away_update,away_datetime_insert,FTR
34056,None,FIFA Club World Cup,2526,2025-07-05 Paris S-G-Bayern Munich,Quarter-finals,NaN,Sat,2025-07-05,12:00:00,Paris S-G,...,Cover,10.0,9.0,31.0,28.27,24.87,FC 25,2025-07-04,2025-07-05 15:42:23.550694,NaN
34055,None,FIFA Club World Cup,2526,2025-07-05 Real Madrid-Dortmund,Quarter-finals,NaN,Sat,2025-07-05,16:00:00,Real Madrid,...,Cover,9.0,8.0,29.0,26.73,23.66,FC 25,2025-07-04,2025-07-05 15:42:23.550694,NaN
34054,7a0bf80a,FIFA Club World Cup,2526,2025-07-01 Real Madrid-Juventus,Round of 16,NaN,Tue,2025-07-01,15:00:00,Real Madrid,...,Cover,10.0,8.0,29.0,24.27,23.79,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0
34053,4997aee4,FIFA Club World Cup,2526,2025-06-26 Juventus-Manchester City,Group stage,3.0,Thu,2025-06-26,15:00:00,Juventus,...,Cover,10.0,10.0,30.0,26.00,24.97,FC 25,2025-05-21,2025-05-22 08:31:00.956585,-1.0
34052,416db7a8,FIFA Club World Cup,2526,2025-06-15 Paris S-G-Atlético Madrid,Group stage,1.0,Sun,2025-06-15,12:00:00,Paris S-G,...,Cover,9.0,9.0,29.0,27.27,26.52,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14,36cd8447,ITA-Serie A,0607,2006-09-09 Fiorentina-Inter,None,1.0,Sat,2006-09-09,None,Fiorentina,...,Cover,10.0,8.0,26.0,28.82,27.77,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,-1.0
15,5c7a478f,ESP-La Liga,0607,2006-09-09 Barcelona-Osasuna,None,2.0,Sat,2006-09-09,None,Barcelona,...,Cover,5.0,1.0,33.0,27.64,24.82,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0
16,ffc11029,ESP-La Liga,0607,2006-09-09 Atlético Madrid-Valencia,None,2.0,Sat,2006-09-09,None,Atlético Madrid,...,Cover,7.0,6.0,33.0,23.55,22.48,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,-1.0
17,85719597,ENG-Premier League,0607,2006-09-09 Bolton-Watford,None,4.0,Sat,2006-09-09,None,Bolton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,1.0


In [74]:
fbref_results_df__sofifa_merged.sort_values(by=["date"], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,away_defence_defender_line,away_defence_domestic_prestige,away_international_prestige,away_players,away_starting_xi_average_age,away_whole_team_average_age,away_fifa_edition,away_update,away_datetime_insert,FTR
34066,None,FIFA Club World Cup,2526,2025-07-05 Real Madrid-Dortmund,Quarter-finals,NaN,Sat,2025-07-05,16:00:00,Real Madrid,...,Cover,9.0,8.0,29.0,26.73,23.66,FC 25,2025-07-04,2025-07-05 15:42:23.550694,NaN
34065,None,FIFA Club World Cup,2526,2025-07-05 Paris S-G-Bayern Munich,Quarter-finals,NaN,Sat,2025-07-05,12:00:00,Paris S-G,...,Cover,10.0,9.0,31.0,28.27,24.87,FC 25,2025-07-04,2025-07-05 15:42:23.550694,NaN
34064,7a0bf80a,FIFA Club World Cup,2526,2025-07-01 Real Madrid-Juventus,Round of 16,NaN,Tue,2025-07-01,15:00:00,Real Madrid,...,Cover,10.0,8.0,29.0,24.27,23.79,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0
34063,4997aee4,FIFA Club World Cup,2526,2025-06-26 Juventus-Manchester City,Group stage,3.0,Thu,2025-06-26,15:00:00,Juventus,...,Cover,10.0,10.0,30.0,26.00,24.97,FC 25,2025-05-21,2025-05-22 08:31:00.956585,-1.0
34062,416db7a8,FIFA Club World Cup,2526,2025-06-15 Paris S-G-Atlético Madrid,Group stage,1.0,Sun,2025-06-15,12:00:00,Paris S-G,...,Cover,9.0,9.0,29.0,27.27,26.52,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14,36cd8447,ITA-Serie A,0607,2006-09-09 Fiorentina-Inter,None,1.0,Sat,2006-09-09,None,Fiorentina,...,Cover,10.0,8.0,26.0,28.82,27.77,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,-1.0
15,5c7a478f,ESP-La Liga,0607,2006-09-09 Barcelona-Osasuna,None,2.0,Sat,2006-09-09,None,Barcelona,...,Cover,5.0,1.0,33.0,27.64,24.82,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0
16,ffc11029,ESP-La Liga,0607,2006-09-09 Atlético Madrid-Valencia,None,2.0,Sat,2006-09-09,None,Atlético Madrid,...,Cover,7.0,6.0,33.0,23.55,22.48,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,-1.0
17,85719597,ENG-Premier League,0607,2006-09-09 Bolton-Watford,None,4.0,Sat,2006-09-09,None,Bolton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,1.0


In [67]:
date_stop=None
ftr_col='FTR'
home_team_goal_col='home_g'
away_team_goal_col='away_g'
date_col='date'
time_col='time'
datetime_col='datetime'
dropna_cols=['home_overall', 'away_overall']

In [68]:
# Fill hom_g and away_g with score value if they are NaN and score is not NaN   
fbref_df_date_filtered_concat = fbref_results_df__sofifa_merged.copy()
fbref_df_date_filtered_concat.loc[:, home_team_goal_col] = fbref_df_date_filtered_concat.apply(lambda x: int(x['score'].split('–')[0]) if pd.isna(x[home_team_goal_col]) and not pd.isna(x['score']) else x[home_team_goal_col], axis=1)
fbref_df_date_filtered_concat.loc[:, away_team_goal_col] = fbref_df_date_filtered_concat.apply(lambda x: int(x['score'].split('–')[1]) if pd.isna(x[away_team_goal_col]) and not pd.isna(x['score']) else x[away_team_goal_col], axis=1)

# Add the full time result column
fbref_df_date_filtered_concat.loc[:, ftr_col] = fbref_df_date_filtered_concat.apply(lambda x: 1 if x[home_team_goal_col] > x[away_team_goal_col] else 0 if x[home_team_goal_col] == x[away_team_goal_col] else -1 if x[home_team_goal_col] < x[away_team_goal_col] else None, axis=1)

# Drop rows with NaN values in the columns of dropna_cols (sofifa cols)
fbref_df_date_filtered_concat_no_nan = fbref_df_date_filtered_concat.dropna(subset=dropna_cols).copy()

fbref_df_date_filtered_concat_no_nan.loc[:, datetime_col] = pd.to_datetime(fbref_df_date_filtered_concat_no_nan[date_col])
fbref_df_date_filtered_concat_no_nan.loc[:, f'{time_col}_'] = fbref_df_date_filtered_concat_no_nan[time_col].apply(lambda x: pd.to_timedelta(x.strftime('%H:%M:%S')) if pd.notnull(x) else pd.to_timedelta('0 days'))
fbref_df_date_filtered_concat_no_nan.loc[:, datetime_col] = fbref_df_date_filtered_concat_no_nan[datetime_col] + fbref_df_date_filtered_concat_no_nan[f'{time_col}_']



In [65]:
fbref_df_date_filtered_concat_no_nan.sort_values(by=['date'], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,away_international_prestige,away_players,away_starting_xi_average_age,away_whole_team_average_age,away_fifa_edition,away_update,away_datetime_insert,FTR,datetime,time_
34066,None,FIFA Club World Cup,2526,2025-07-05 Real Madrid-Dortmund,Quarter-finals,NaN,Sat,2025-07-05,16:00:00,Real Madrid,...,8.0,29.0,26.73,23.66,FC 25,2025-07-04,2025-07-05 15:42:23.550694,NaN,2025-07-05 16:00:00,0 days 16:00:00
34065,None,FIFA Club World Cup,2526,2025-07-05 Paris S-G-Bayern Munich,Quarter-finals,NaN,Sat,2025-07-05,12:00:00,Paris S-G,...,9.0,31.0,28.27,24.87,FC 25,2025-07-04,2025-07-05 15:42:23.550694,NaN,2025-07-05 12:00:00,0 days 12:00:00
34064,7a0bf80a,FIFA Club World Cup,2526,2025-07-01 Real Madrid-Juventus,Round of 16,NaN,Tue,2025-07-01,15:00:00,Real Madrid,...,8.0,29.0,24.27,23.79,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0,2025-07-01 15:00:00,0 days 15:00:00
34063,4997aee4,FIFA Club World Cup,2526,2025-06-26 Juventus-Manchester City,Group stage,3.0,Thu,2025-06-26,15:00:00,Juventus,...,10.0,30.0,26.00,24.97,FC 25,2025-05-21,2025-05-22 08:31:00.956585,-1.0,2025-06-26 15:00:00,0 days 15:00:00
34062,416db7a8,FIFA Club World Cup,2526,2025-06-15 Paris S-G-Atlético Madrid,Group stage,1.0,Sun,2025-06-15,12:00:00,Paris S-G,...,9.0,29.0,27.27,26.52,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0,2025-06-15 12:00:00,0 days 12:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,dc45f23c,ENG-Premier League,0607,2006-09-09 Manchester Utd-Tottenham,None,4.0,Sat,2006-09-09,None,Manchester Utd,...,7.0,31.0,24.55,23.74,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0,2006-09-09 00:00:00,0 days 00:00:00
6,880e137f,ENG-Premier League,0607,2006-09-09 Everton-Liverpool,None,4.0,Sat,2006-09-09,None,Everton,...,9.0,33.0,26.27,23.76,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0,2006-09-09 00:00:00,0 days 00:00:00
15,5c7a478f,ESP-La Liga,0607,2006-09-09 Barcelona-Osasuna,None,2.0,Sat,2006-09-09,None,Barcelona,...,1.0,33.0,27.64,24.82,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0,2006-09-09 00:00:00,0 days 00:00:00
16,ffc11029,ESP-La Liga,0607,2006-09-09 Atlético Madrid-Valencia,None,2.0,Sat,2006-09-09,None,Atlético Madrid,...,6.0,33.0,23.55,22.48,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,-1.0,2006-09-09 00:00:00,0 days 00:00:00


In [61]:
date_stop

datetime.datetime(2025, 7, 5, 17, 48, 53, 667928)

In [63]:
fbref_df_date_filtered_concat_no_nan.sort_values(by=['date'], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,away_international_prestige,away_players,away_starting_xi_average_age,away_whole_team_average_age,away_fifa_edition,away_update,away_datetime_insert,FTR,datetime,time_
34064,7a0bf80a,FIFA Club World Cup,2526,2025-07-01 Real Madrid-Juventus,Round of 16,NaN,Tue,2025-07-01,15:00:00,Real Madrid,...,8.0,29.0,24.27,23.79,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0,2025-07-01 15:00:00,0 days 15:00:00
34063,4997aee4,FIFA Club World Cup,2526,2025-06-26 Juventus-Manchester City,Group stage,3.0,Thu,2025-06-26,15:00:00,Juventus,...,10.0,30.0,26.00,24.97,FC 25,2025-05-21,2025-05-22 08:31:00.956585,-1.0,2025-06-26 15:00:00,0 days 15:00:00
34062,416db7a8,FIFA Club World Cup,2526,2025-06-15 Paris S-G-Atlético Madrid,Group stage,1.0,Sun,2025-06-15,12:00:00,Paris S-G,...,9.0,29.0,27.27,26.52,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0,2025-06-15 12:00:00,0 days 12:00:00
34051,15559cff,ENG-Premier League,2425,2025-05-25 Ipswich Town-West Ham,None,38.0,Sun,2025-05-25,16:00:00,Ipswich Town,...,6.0,28.0,27.00,26.82,FC 25,2025-05-21,2025-05-22 08:31:00.956585,-1.0,2025-05-25 16:00:00,0 days 16:00:00
34042,75ae7628,ITA-Serie A,2425,2025-05-25 Atalanta-Parma,None,38.0,Sun,2025-05-25,20:45:00,Atalanta,...,3.0,31.0,22.18,23.52,FC 25,2025-05-21,2025-05-22 08:31:00.956585,-1.0,2025-05-25 20:45:00,0 days 20:45:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,dc45f23c,ENG-Premier League,0607,2006-09-09 Manchester Utd-Tottenham,None,4.0,Sat,2006-09-09,None,Manchester Utd,...,7.0,31.0,24.55,23.74,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0,2006-09-09 00:00:00,0 days 00:00:00
6,880e137f,ENG-Premier League,0607,2006-09-09 Everton-Liverpool,None,4.0,Sat,2006-09-09,None,Everton,...,9.0,33.0,26.27,23.76,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0,2006-09-09 00:00:00,0 days 00:00:00
15,5c7a478f,ESP-La Liga,0607,2006-09-09 Barcelona-Osasuna,None,2.0,Sat,2006-09-09,None,Barcelona,...,1.0,33.0,27.64,24.82,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0,2006-09-09 00:00:00,0 days 00:00:00
16,ffc11029,ESP-La Liga,0607,2006-09-09 Atlético Madrid-Valencia,None,2.0,Sat,2006-09-09,None,Atlético Madrid,...,6.0,33.0,23.55,22.48,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,-1.0,2006-09-09 00:00:00,0 days 00:00:00


In [66]:
date_col

'date'

In [62]:
fbref_df_date_filtered_concat_no_nan[fbref_df_date_filtered_concat_no_nan['date']>= date_stop].sort_values(by=['date'], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,away_international_prestige,away_players,away_starting_xi_average_age,away_whole_team_average_age,away_fifa_edition,away_update,away_datetime_insert,FTR,datetime,time_


In [69]:
# Drop rows with NaN values in the ftr_col but keep future matches
date_stop = datetime.datetime.now().replace(hour=0, minute=0, second=0, microsecond=0) if not date_stop else date_stop
rule_is_null = (fbref_df_date_filtered_concat_no_nan[ftr_col].isnull())
rule_is_future_match = (fbref_df_date_filtered_concat_no_nan[date_col] >= date_stop)
rule_remove_nan_and_keep_future_matches = ~rule_is_null | rule_is_future_match
fbref_df_date_filtered_concat_no_nan = fbref_df_date_filtered_concat_no_nan[rule_remove_nan_and_keep_future_matches]


In [70]:
fbref_df_date_filtered_concat_no_nan.sort_values(by=['date'], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,away_international_prestige,away_players,away_starting_xi_average_age,away_whole_team_average_age,away_fifa_edition,away_update,away_datetime_insert,FTR,datetime,time_
34066,None,FIFA Club World Cup,2526,2025-07-05 Real Madrid-Dortmund,Quarter-finals,NaN,Sat,2025-07-05,16:00:00,Real Madrid,...,8.0,29.0,26.73,23.66,FC 25,2025-07-04,2025-07-05 15:42:23.550694,NaN,2025-07-05 16:00:00,0 days 16:00:00
34065,None,FIFA Club World Cup,2526,2025-07-05 Paris S-G-Bayern Munich,Quarter-finals,NaN,Sat,2025-07-05,12:00:00,Paris S-G,...,9.0,31.0,28.27,24.87,FC 25,2025-07-04,2025-07-05 15:42:23.550694,NaN,2025-07-05 12:00:00,0 days 12:00:00
34064,7a0bf80a,FIFA Club World Cup,2526,2025-07-01 Real Madrid-Juventus,Round of 16,NaN,Tue,2025-07-01,15:00:00,Real Madrid,...,8.0,29.0,24.27,23.79,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0,2025-07-01 15:00:00,0 days 15:00:00
34063,4997aee4,FIFA Club World Cup,2526,2025-06-26 Juventus-Manchester City,Group stage,3.0,Thu,2025-06-26,15:00:00,Juventus,...,10.0,30.0,26.00,24.97,FC 25,2025-05-21,2025-05-22 08:31:00.956585,-1.0,2025-06-26 15:00:00,0 days 15:00:00
34062,416db7a8,FIFA Club World Cup,2526,2025-06-15 Paris S-G-Atlético Madrid,Group stage,1.0,Sun,2025-06-15,12:00:00,Paris S-G,...,9.0,29.0,27.27,26.52,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0,2025-06-15 12:00:00,0 days 12:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,dc45f23c,ENG-Premier League,0607,2006-09-09 Manchester Utd-Tottenham,None,4.0,Sat,2006-09-09,None,Manchester Utd,...,7.0,31.0,24.55,23.74,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0,2006-09-09 00:00:00,0 days 00:00:00
6,880e137f,ENG-Premier League,0607,2006-09-09 Everton-Liverpool,None,4.0,Sat,2006-09-09,None,Everton,...,9.0,33.0,26.27,23.76,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0,2006-09-09 00:00:00,0 days 00:00:00
15,5c7a478f,ESP-La Liga,0607,2006-09-09 Barcelona-Osasuna,None,2.0,Sat,2006-09-09,None,Barcelona,...,1.0,33.0,27.64,24.82,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0,2006-09-09 00:00:00,0 days 00:00:00
16,ffc11029,ESP-La Liga,0607,2006-09-09 Atlético Madrid-Valencia,None,2.0,Sat,2006-09-09,None,Atlético Madrid,...,6.0,33.0,23.55,22.48,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,-1.0,2006-09-09 00:00:00,0 days 00:00:00


In [35]:
fbref_df_date_filtered_concat_no_nan.sort_values(by=['date'], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,away_defence_defender_line,away_defence_domestic_prestige,away_international_prestige,away_players,away_starting_xi_average_age,away_whole_team_average_age,away_fifa_edition,away_update,away_datetime_insert,FTR
34066,None,FIFA Club World Cup,2526,2025-07-05 Real Madrid-Dortmund,Quarter-finals,NaN,Sat,2025-07-05,16:00:00,Real Madrid,...,Cover,9.0,8.0,29.0,26.73,23.66,FC 25,2025-07-04,2025-07-05 15:42:23.550694,NaN
34065,None,FIFA Club World Cup,2526,2025-07-05 Paris S-G-Bayern Munich,Quarter-finals,NaN,Sat,2025-07-05,12:00:00,Paris S-G,...,Cover,10.0,9.0,31.0,28.27,24.87,FC 25,2025-07-04,2025-07-05 15:42:23.550694,NaN
34064,7a0bf80a,FIFA Club World Cup,2526,2025-07-01 Real Madrid-Juventus,Round of 16,NaN,Tue,2025-07-01,15:00:00,Real Madrid,...,Cover,10.0,8.0,29.0,24.27,23.79,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0
34063,4997aee4,FIFA Club World Cup,2526,2025-06-26 Juventus-Manchester City,Group stage,3.0,Thu,2025-06-26,15:00:00,Juventus,...,Cover,10.0,10.0,30.0,26.00,24.97,FC 25,2025-05-21,2025-05-22 08:31:00.956585,-1.0
34062,416db7a8,FIFA Club World Cup,2526,2025-06-15 Paris S-G-Atlético Madrid,Group stage,1.0,Sun,2025-06-15,12:00:00,Paris S-G,...,Cover,9.0,9.0,29.0,27.27,26.52,FC 25,2025-05-21,2025-05-22 08:31:00.956585,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,dc45f23c,ENG-Premier League,0607,2006-09-09 Manchester Utd-Tottenham,None,4.0,Sat,2006-09-09,None,Manchester Utd,...,Cover,7.0,7.0,31.0,24.55,23.74,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0
6,880e137f,ENG-Premier League,0607,2006-09-09 Everton-Liverpool,None,4.0,Sat,2006-09-09,None,Everton,...,Cover,9.0,9.0,33.0,26.27,23.76,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0
15,5c7a478f,ESP-La Liga,0607,2006-09-09 Barcelona-Osasuna,None,2.0,Sat,2006-09-09,None,Barcelona,...,Cover,5.0,1.0,33.0,27.64,24.82,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,1.0
16,ffc11029,ESP-La Liga,0607,2006-09-09 Atlético Madrid-Valencia,None,2.0,Sat,2006-09-09,None,Atlético Madrid,...,Cover,7.0,6.0,33.0,23.55,22.48,FIFA 07,2006-08-30,2024-09-24 08:59:36.003057,-1.0


In [21]:
fbref_results_df__sofifa_merged__data_formated.sort_values(by=["date"], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,avg_home_team_goals_season_to_date_before_match,avg_away_team_goals_season_to_date_before_match,glicko2_home_before,glicko2_away_before,glicko2_rd_home_before,glicko2_rd_away_before,glicko2_vol_home_before,glicko2_vol_away_before,trueskill_home_before,trueskill_away_before
34064,7a0bf80a,FIFA Club World Cup,2526,2025-07-01 Real Madrid-Juventus,Round of 16,NaN,Tue,2025-07-01,15:00:00,Real Madrid,...,0.0,2.000000,1726.161467,1681.404975,26.839210,27.220442,0.059423,0.059386,26.553652,25.389222
34063,4997aee4,FIFA Club World Cup,2526,2025-06-26 Juventus-Manchester City,Group stage,3.0,Thu,2025-06-26,15:00:00,Juventus,...,0.0,0.000000,1683.322414,1717.541004,26.771348,26.038646,0.059386,0.059413,25.463776,26.658899
34062,416db7a8,FIFA Club World Cup,2526,2025-06-15 Paris S-G-Atlético Madrid,Group stage,1.0,Sun,2025-06-15,12:00:00,Paris S-G,...,0.0,0.000000,1737.408535,1668.043230,26.132319,27.214162,0.059370,0.059387,27.883608,25.182414
34051,15559cff,ENG-Premier League,2425,2025-05-25 Ipswich Town-West Ham,None,38.0,Sun,2025-05-25,16:00:00,Ipswich Town,...,1.0,1.162162,1462.243556,1502.504945,28.629302,28.589834,0.059970,0.059449,17.640349,22.238778
34042,75ae7628,ITA-Serie A,2425,2025-05-25 Atalanta-Parma,None,38.0,Sun,2025-05-25,20:45:00,Atalanta,...,2.0,1.108108,1621.573223,1443.155898,28.075948,27.459647,0.059421,0.059650,25.125309,20.827517
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,dc45f23c,ENG-Premier League,0607,2006-09-09 Manchester Utd-Tottenham,None,4.0,Sat,2006-09-09,None,Manchester Utd,...,0.0,0.000000,1500.000000,1500.000000,350.000000,350.000000,0.060000,0.060000,0.000000,0.000000
6,880e137f,ENG-Premier League,0607,2006-09-09 Everton-Liverpool,None,4.0,Sat,2006-09-09,None,Everton,...,0.0,0.000000,1500.000000,1500.000000,350.000000,350.000000,0.060000,0.060000,0.000000,0.000000
15,5c7a478f,ESP-La Liga,0607,2006-09-09 Barcelona-Osasuna,None,2.0,Sat,2006-09-09,None,Barcelona,...,0.0,0.000000,1500.000000,1500.000000,350.000000,350.000000,0.060000,0.060000,0.000000,0.000000
16,ffc11029,ESP-La Liga,0607,2006-09-09 Atlético Madrid-Valencia,None,2.0,Sat,2006-09-09,None,Atlético Madrid,...,0.0,0.000000,1500.000000,1500.000000,350.000000,350.000000,0.060000,0.060000,0.000000,0.000000


In [16]:
fbref_results_df__sofifa_merged__data_formated__signals_added__train.sort_values(by=["date"], ascending=False)

,game_id,league,season,game,round,week,day,date,time,home_team,...,away_chance_creation_shooting_Lots,away_chance_creation_shooting_Normal,away_chance_creation_positioning_Organised,away_defence_aggression_Double,away_defence_aggression_Press,away_defence_pressure_High,away_defence_pressure_Medium,away_defence_team_width_Normal,away_defence_team_width_Wide,away_defence_defender_line_Offside trap
34064,7a0bf80a,FIFA Club World Cup,2526,2025-07-01 Real Madrid-Juventus,Round of 16,NaN,Tue,2025-07-01,15:00:00,Real Madrid,...,False,False,True,False,False,False,False,False,False,False
34063,4997aee4,FIFA Club World Cup,2526,2025-06-26 Juventus-Manchester City,Group stage,3.0,Thu,2025-06-26,15:00:00,Juventus,...,False,False,True,False,False,False,False,False,False,False
34062,416db7a8,FIFA Club World Cup,2526,2025-06-15 Paris S-G-Atlético Madrid,Group stage,1.0,Sun,2025-06-15,12:00:00,Paris S-G,...,False,False,True,False,False,False,False,False,False,False
34051,15559cff,ENG-Premier League,2425,2025-05-25 Ipswich Town-West Ham,None,38.0,Sun,2025-05-25,16:00:00,Ipswich Town,...,False,False,True,False,False,False,False,False,False,False
34042,75ae7628,ITA-Serie A,2425,2025-05-25 Atalanta-Parma,None,38.0,Sun,2025-05-25,20:45:00,Atalanta,...,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,dc45f23c,ENG-Premier League,0607,2006-09-09 Manchester Utd-Tottenham,None,4.0,Sat,2006-09-09,None,Manchester Utd,...,False,False,True,False,False,False,False,False,False,False
6,880e137f,ENG-Premier League,0607,2006-09-09 Everton-Liverpool,None,4.0,Sat,2006-09-09,None,Everton,...,False,False,True,False,False,False,False,False,False,False
15,5c7a478f,ESP-La Liga,0607,2006-09-09 Barcelona-Osasuna,None,2.0,Sat,2006-09-09,None,Barcelona,...,False,False,True,False,False,False,False,False,False,False
16,ffc11029,ESP-La Liga,0607,2006-09-09 Atlético Madrid-Valencia,None,2.0,Sat,2006-09-09,None,Atlético Madrid,...,False,False,True,False,False,False,False,False,False,False


In [ ]:

    #### Train and test the model and infer the results ####
    start_train_test_inference = time.time()
    try:
        train_test_metrics, fbref_results_df__sofifa_merged__data_formated__signals_added__infered = test_model_and_infer(
            fbref_results_df__sofifa_merged__data_formated__signals_added__train, 
            fbref_results_df__sofifa_merged__data_formated__signals_added__infer
        )
        logger.info(f"Model training and inference completed successfully completed in {time.time() - start_train_test_inference} seconds, accuray on train test : {train_test_metrics}")
    except Exception as e:
        logger.error(f"Error during model training and inference: {e}")
        raise


    #### Insert the results into the database ####
    start_insert_results = time.time()
    try:
        datetime_inference = insert_results_to_db(engine, fbref_results_df__sofifa_merged__data_formated__signals_added__infered, DB_TN_MODELS_RESULTS)
        matchs_infered = fbref_results_df__sofifa_merged__data_formated__signals_added__infered.shape[0]
        logger.info(f"{matchs_infered} results inserted into the database successfully, completed in {time.time() - start_insert_results} seconds")
    except Exception as e:
        logger.error(f"Error inserting results into the database: {e}")
        raise

    #### Calculate some metrics ####
    try:
        df_infered = fbref_results_df__sofifa_merged__data_formated__signals_added__infered
        nb_matches_infered = df_infered.shape[0]
        df_infered['time_match'] = df_infered['time_match'].replace([None, np.nan], '00:00:00')
        df_infered['datetime_match'] = pd.to_datetime(df_infered['date_match'].astype(str) + ' ' + df_infered['time_match'].astype(str))
        first_match_name = df_infered[df_infered['datetime_match'] == df_infered['datetime_match'].min()].iloc[0]['game']
        last_match_name = df_infered[df_infered['datetime_match'] == df_infered['datetime_match'].max()].iloc[0]['game']
        logger.info(f"Pipeline completed in {time.time() - start_pipeline} seconds")
        return train_test_metrics, fbref_results_df__sofifa_merged__data_formated__signals_added__infered, nb_matches_infered, first_match_name, last_match_name, datetime_inference
    except Exception as e:
        logger.warning(f"Error calculating metrics: {e}")
        return train_test_metrics, fbref_results_df__sofifa_merged__data_formated__signals_added__infered, None, None, None, datetime_inference

if __name__ == "__main__":
    start_time = time.time()

    args = argparse.ArgumentParser()
    args.add_argument("--date_stop", type=str, default=None)
    args = args.parse_args()

    date_stop = None
    if args.date_stop:
        try:
            date_stop = datetime.datetime.strptime(args.date_stop, "%Y-%m-%d %H:%M:%S")
            logger.info(f"date_stop parameter parsed successfully: {date_stop}")
        except ValueError as e:
            logger.error(f"Error parsing date_stop parameter: {e}")
            raise

    try:
        train_test_metrics, df_infered, nb_matches_infered, first_match_name, last_match_name, datetime_inference = infer__RSF_PR_LR__pipeline(date_stop=date_stop)
        end_time = time.time()
        duration = end_time - start_time
        logger.info(f"Pipeline executed successfully in {duration:2f} seconds \n\n")
    except Exception as e:
        logger.error(f"Error executing the pipeline: {e} \n\n")
        raise
